In [1]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
!pip install -U transformers accelerate sentencepiece gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 7.1 MB/s eta 0:00:00
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 2.5.0
    Uninstalling gradio_client-2.5.0:
      Successfully uninstalled gradio_client-2.5.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1
  Attempting uninstall: gradio
    Found existing installation: gradio 6.20.0
    Uninstalling gradio-6.20.0:
      Successfully uninstalled gradio-6.20.0


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain.memory import ConversationBufferMemory
import gradio as gr

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import gradio as gr

In [5]:
from huggingface_hub import login

login()

In [6]:
model_name = "meta-llama/Llama-2-7b-chat-hf"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [7]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

In [8]:
prompt = "What is artificial intelligence?"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is artificial intelligence?

Artificial intelligence (AI) refers to the development of computer systems able to perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation. AI systems use machine learning, deep learning, and natural language processing to analyze data and make predictions or decisions.

There are several types of AI, including:

1. Narrow or weak AI: This type of AI is designed to perform a


In [9]:
def chatbot_response(user_input):

    inputs = tokenizer(
        user_input,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
        top_p=0.9
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

In [10]:
print(chatbot_response("Explain machine learning in simple words."))

[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain machine learning in simple words.
Machine learning is a type of artificial intelligence that allows a computer to learn and improve its performance on a task without explicitly being programmed for that task. In other words, the computer can learn from data and experience, and improve its performance over time.

Imagine you are trying to teach a computer to recognize different types of flowers. You would need to show the computer many pictures of each type of flower, and tell it which one is which. Once the computer has learned to recognize each type of flower, it can then be shown new pictures of flowers it hasn't seen before, and it can use its knowledge to identify the type of flower.

This is similar to how a child learns to recognize different types of objects


In [15]:
import gradio as gr

def chat(message, history):
    response = chatbot_response(message)
    return response

demo = gr.ChatInterface(
    fn=chat,
    title="🤖 Llama 2 Chatbot",
    description="Ask me anything!"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://75f9f289b9e6612089.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [16]:
def chat(message, history):
    conversation = ""

    # Add previous conversation
    for item in history:
        if isinstance(item, dict):
            role = item["role"]
            content = item["content"]

            if role == "user":
                conversation += f"User: {content}\n"
            elif role == "assistant":
                conversation += f"Assistant: {content}\n"
        else:
            # For older Gradio history format
            user_message, assistant_message = item
            conversation += f"User: {user_message}\n"
            conversation += f"Assistant: {assistant_message}\n"

    # Add current message
    conversation += f"User: {message}\nAssistant:"

    inputs = tokenizer(
        conversation,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
        top_p=0.9
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Remove the prompt from the displayed answer
    if "Assistant:" in response:
        response = response.split("Assistant:")[-1].strip()

    return response

In [17]:
demo = gr.ChatInterface(
    fn=chat,
    title="🤖 Llama 2 Chatbot",
    description="A conversational AI chatbot powered by Llama 2"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cab4371ca1f1514215.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [19]:
import gradio as gr

demo = gr.ChatInterface(
    fn=chat,
    title="🤖 Llama 2 AI Chatbot",
    description="Ask questions and have a conversation with Llama 2.",
    examples=[
        "Explain artificial intelligence in simple words.",
        "What is machine learning?",
        "Give me 5 interesting facts about space.",
        "Help me write a Python program."
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5d0963cc3fde71b272.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [20]:
def chat(message, history):

    system_prompt = """
You are a friendly and helpful AI assistant.

Your responsibilities:
- Answer questions clearly and accurately.
- Explain difficult topics in simple language.
- Give step-by-step explanations when needed.
- Be polite and conversational.
- If you don't know something, say so instead of making up information.
"""

    conversation = system_prompt + "\n\n"

    # Add previous conversation
    for item in history:

        if isinstance(item, dict):

            if item["role"] == "user":
                conversation += f"User: {item['content']}\n"

            elif item["role"] == "assistant":
                conversation += f"Assistant: {item['content']}\n"

        else:
            # Compatibility with older Gradio history format
            user_message, assistant_message = item
            conversation += f"User: {user_message}\n"
            conversation += f"Assistant: {assistant_message}\n"

    # Add current question
    conversation += f"User: {message}\nAssistant:"

    inputs = tokenizer(
        conversation,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        top_p=0.9
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    if "Assistant:" in response:
        response = response.split("Assistant:")[-1].strip()

    return response

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

In [2]:
login()

In [3]:
model_name = "meta-llama/Llama-2-7b-chat-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded successfully!


In [4]:
prompt = "What is artificial intelligence? Explain in simple words."

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True
    )

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is artificial intelligence? Explain in simple words.
Artificial intelligence (AI) is the ability of a computer or machine to perform tasks that would normally require human intelligence, such as understanding language, recognizing images, making decisions, and solving problems.
In simple terms, AI is when a computer or machine can think and act like a human, but faster and more accurately. It can do things like:

* Understand and respond to voice commands
* Recognize faces and objects in photos and videos



In [5]:
def chatbot_response(user_input):

    inputs = tokenizer(
        user_input,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_p=0.9
        )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

In [6]:
print(chatbot_response("Explain machine learning in simple words."))

[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain machine learning in simple words.
Machine learning is a way for computers to learn from data without being explicitly programmed. It involves using algorithms and statistical models to enable computers to learn from data, make decisions, and improve their performance over time.

In simple terms, machine learning is a type of artificial intelligence that enables computers to learn and improve their performance on a task without being explicitly programmed for that task. It works by analyzing large amounts of data and identifying patterns, which can then be used to make predictions or decisions.

For example, a machine learning algorithm can be trained on a dataset of images of dogs and cats, and then use that information to identify new images of dogs and cats that it has not seen before. Over


In [7]:
import gradio as gr

def chat(message, history):
    response = chatbot_response(message)
    return response

demo = gr.ChatInterface(
    fn=chat,
    title="🤖 Llama 2 AI Chatbot",
    description="Ask questions and have a conversation with Llama 2.",
    examples=[
        "Explain artificial intelligence in simple words.",
        "What is machine learning?",
        "Tell me an interesting fact about space.",
        "Explain Python to a beginner."
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f995be9e03c816da30.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
def chat(message, history):
    conversation = ""

    # Include previous messages
    for item in history:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content", "")

            if role == "user":
                conversation += f"User: {content}\n"
            elif role == "assistant":
                conversation += f"Assistant: {content}\n"

        elif isinstance(item, (list, tuple)) and len(item) == 2:
            conversation += f"User: {item[0]}\n"
            conversation += f"Assistant: {item[1]}\n"

    # Add the current message
    conversation += f"User: {message}\nAssistant:"

    inputs = tokenizer(
        conversation,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_p=0.9
        )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Remove the prompt from the displayed response
    if "Assistant:" in response:
        response = response.split("Assistant:")[-1].strip()

    return response

In [9]:
import gradio as gr

demo = gr.ChatInterface(
    fn=chat,
    title="🤖 Llama 2 AI Chatbot",
    description="Ask questions and have a conversation with Llama 2.",
    examples=[
        "Explain artificial intelligence in simple words.",
        "What is machine learning?",
        "Tell me an interesting fact about space.",
        "Explain Python to a beginner."
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://081ad37307f5a43a78.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
